In [ ]:
"""Pure radial diffusion into a silicon cylinder:


The model is dimensionless on x = r/R in [0, 1] and (tau) = D*t/R**2:
u_tau = (1/x) * (x*u_x)_x,  u(x, 0) = 0,  u(1, tau) = 1.
Here u = (c-c0)/(cmax-c0). At x=0, regularity gives the natural symmetry
condition. 

what each function does:
- IntervalMesh: creates a 1D mesh of the interval [0, 1]
- FunctionSpace: creates a function space on the mesh, in this case continuous Galerkin
- spatialCoordinate: gets the spatial coordinates of the mesh points
"""


from firedrake import * 

#mesh and concentration variable

mesh = IntervalMesh(80, 1)
V = FunctionSpace(mesh, "CG", 1) #continuous Galerkin space of degree 1, straight lines joining the nodes
rho = SpatialCoordinate(mesh)[0] #radial coordinate at each point in the mesh

q_p = Function(V) #concentration variable at previous time step
q = Function(V) #concentration variable at current time step
q_p.assign(0) #initial condition: concentration is zero at all points in the mesh


Coefficient(WithGeometry(FunctionSpace(<firedrake.mesh.MeshTopology object at 0xffff5e038bf0>, FiniteElement('Lagrange', interval, 1), name=None), Mesh(VectorElement(FiniteElement('Lagrange', interval, 1), dim=1), 15)), 35)

In [ ]:
dtau = 0.01 #time step size
trial = TrialFunction(V) #place holder for the new q
test = TestFunction(V) #weight function for the weak form

lhs = (
    rho * trial * test /dtau
    + rho * dot(grad(trial), grad(test))
)*dx #weak form of the equation integrated over the domain, multiplied by the test function and the radial coordinate
rhs = (
    rho * q_p * test / dtau *dx #rhs is the right hand side of the equation
)

bc_s = DirichletBC(V, 1.0, 2) #boundary condition at x=1, concentration q is 1
problem = LinearVariationalProblem(lhs, rhs, q, bcs=bc_s) #set up the problem to be solved
solver = LinearVariationalSolver(problem) #set up the solver for the problem
#Advanced the solution in time


In [ ]:
t = [0]
mean = [0]

for i in range(1, 51):
    solver.solve() #solve the problem for the current time step
    q_p.assign(q) #update the previous concentration to the current concentration for the next time step


    mean_q = assemble(2* q * rho*dx) #compute the mean concentration over the domain
    t.append((i)*dtau) #append the current time to the time list
    mean.append(mean_q) #append the mean concentration to the mean list
    

#plot the final concentration profile over the time domain
import matplotlib.pyplot as plt

plt.plot(t, mean)
plt.xlabel('Dimensionless time (tau)')
plt.ylabel('Mean concentration (u)')
plt.title('Mean concentration vs time for pure radial diffusion')
plt.ylim(0, 1.05)
plt.grid(True)
plt.show()
